In [1]:
# import essential
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [1]:
# Import required modules for noise correlation analysis
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import xarray as xr
from ephys_dimension_reduction_tdr import tdr_from_psth
from create_psth import load_zarr
from general_utils import smart_read_csv

def calculate_noise_correlation(
    psth_da,
    latent_var_values,
    trial_numbers=None,
    time_window=[-1, 0],
    align="go_cue",
    include_trials=None
):
    """
    Calculate noise correlation between units after regressing out confounds.
    
    Parameters:
    -----------
    psth_da : xr.DataArray
        PSTH data with dimensions [trial, unit, time]
    latent_var_values : array-like
        Values of the latent variable for each trial
    trial_numbers : array-like, optional
        Trial numbers for regression. If None, uses sequential numbering
    time_window : list, default [-1, 0]
        Time window relative to alignment event [start, end] in seconds
    align : str, default "go_cue"
        Alignment event
    include_trials : array-like, optional
        Indices of trials to include
        
    Returns:
    --------
    noise_corr_matrix : np.ndarray
        Correlation matrix between units (n_units x n_units)
    residuals : np.ndarray
        Residual firing rates after regression (n_trials x n_units x n_timepoints)
    """
    
    # Select time window
    time_coords = psth_da.coords["time"].values
    time_mask = (time_coords >= time_window[0]) & (time_coords <= time_window[1])
    psth_windowed = psth_da.sel(time=time_mask)
    
    # Filter trials if specified
    if include_trials is not None:
        trial_mask = np.isin(psth_da.coords["trial"].values, include_trials)
        psth_windowed = psth_windowed.isel(trial=trial_mask)
        latent_var_values = np.array(latent_var_values)[trial_mask]
        if trial_numbers is not None:
            trial_numbers = np.array(trial_numbers)[trial_mask]
    
    # Get data dimensions
    n_trials, n_units, n_timepoints = psth_windowed.shape
    
    # Create trial numbers if not provided
    if trial_numbers is None:
        trial_numbers = np.arange(n_trials)
    
    # Prepare data for regression: concatenate time points for each trial-unit pair
    # Shape: (n_trials * n_timepoints, n_units)
    firing_rates = psth_windowed.values.reshape(n_trials * n_timepoints, n_units)
    
    # Create regressors - repeat for each time point within trials
    latent_regressors = np.repeat(latent_var_values, n_timepoints)
    trial_regressors = np.repeat(trial_numbers, n_timepoints)
    
    # Create design matrix
    X = np.column_stack([
        np.ones(len(latent_regressors)),  # intercept
        latent_regressors,                # latent variable
        trial_regressors                  # trial number
    ])
    
    # Initialize residuals array
    residuals = np.zeros_like(firing_rates)
    
    # For each unit, regress out confounds
    for unit_idx in range(n_units):
        y = firing_rates[:, unit_idx]
        
        # Fit linear regression
        reg = LinearRegression(fit_intercept=False)
        reg.fit(X, y)
        
        # Calculate residuals (observed - predicted)
        y_pred = reg.predict(X)
        residuals[:, unit_idx] = y - y_pred
    
    # Reshape residuals back to (n_trials, n_units, n_timepoints)
    residuals_reshaped = residuals.reshape(n_trials, n_units, n_timepoints)
    
    # Calculate mean-centered residuals for correlation
    # Concatenate across trials and time for each unit
    unit_residuals_concat = []
    for unit_idx in range(n_units):
        unit_data = residuals_reshaped[:, unit_idx, :].flatten()
        # Subtract mean across all time points and trials for this unit
        unit_data_centered = unit_data - np.mean(unit_data)
        unit_residuals_concat.append(unit_data_centered)
    
    unit_residuals_concat = np.array(unit_residuals_concat)  # Shape: (n_units, n_trials * n_timepoints)
    
    # Calculate noise correlation matrix
    noise_corr_matrix = np.corrcoef(unit_residuals_concat)
    
    return noise_corr_matrix, residuals_reshaped, unit_residuals_concat

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Load session summary data (assuming this exists in your environment)
# You'll need to define session_summary_df or load it from your data source
# For now, I'll create a placeholder - replace this with your actual data loading

# Example session configuration
session_name = "your_session_name"  # Replace with actual session name
binsize = '0.2'
align = "go_cue"
time_window0 = [-1, 0]
latent_var = 'ForagingCompareThreshold-value-1'

# Load PSTH data and behavior data
# Note: Update these paths to match your actual data locations
try:
    psth_da = load_zarr(f"/root/capsule/scratch/psth_results/{session_name}_{binsize}s.zarr")
    df = smart_read_csv(f"/root/capsule/scratch/behavior_summary/behavior_summary-{session_name}.csv")
    
    print(f"Loaded PSTH data shape: {psth_da.shape}")
    print(f"Loaded behavior data shape: {df.shape}")
    
    # Get trial IDs to include
    keep_ids = np.asarray(df['response_trials'][0], dtype=int)
    print(f"Number of trials to include: {len(keep_ids)}")
    
    # Get latent variable values
    latent_values = df[latent_var][0]
    print(f"Latent variable range: {np.min(latent_values)} to {np.max(latent_values)}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please ensure the data paths and session_name are correct")

In [ ]:
# Using your provided variables to calculate noise correlation
# (Replace the above cell with your actual data loading code)

# Assuming you have already loaded:
# session_summary_df, session_name, psth_da, df, keep_ids, etc.
# as shown in your original code

# For demonstration, let me show how to use the data you provided:

# Calculate noise correlation for all units
print("Calculating noise correlation for all units...")
try:
    # Extract trial numbers for regression
    trial_numbers = keep_ids
    
    # Calculate noise correlation
    noise_corr_all, residuals_all, unit_residuals_all = calculate_noise_correlation(
        psth_da=psth_da,
        latent_var_values=df[latent_var][0],
        trial_numbers=trial_numbers,
        time_window=time_window0,
        align=align,
        include_trials=keep_ids
    )
    
    print(f"Noise correlation matrix shape (all units): {noise_corr_all.shape}")
    print(f"Mean absolute correlation (off-diagonal): {np.mean(np.abs(noise_corr_all[~np.eye(noise_corr_all.shape[0], dtype=bool)])):.4f}")
    
    # Save results
    np.save(f"/root/capsule/scratch/long_time_scale_correlation/noise_corr_all_{session_name}_{latent_var}_timewindow_{time_window0[0]}_{time_window0[1]}.npy", 
            noise_corr_all)
    
except NameError as e:
    print(f"Some variables are not defined: {e}")
    print("Please run your data loading code first, then execute this cell")

In [ ]:
# Calculate noise correlation for positive units
print("Calculating noise correlation for positive units...")
try:
    # Filter for positive units
    positive_unit_index = row["positive_unit_index"].iloc[0]
    mask_positive = np.isin(psth_da.coords["unit_index"].values, positive_unit_index)
    psth_da_positive = psth_da.isel(unit=mask_positive)
    
    noise_corr_positive, residuals_positive, unit_residuals_positive = calculate_noise_correlation(
        psth_da=psth_da_positive,
        latent_var_values=df[latent_var][0],
        trial_numbers=trial_numbers,
        time_window=time_window0,
        align=align,
        include_trials=keep_ids
    )
    
    print(f"Noise correlation matrix shape (positive units): {noise_corr_positive.shape}")
    print(f"Mean absolute correlation (off-diagonal, positive): {np.mean(np.abs(noise_corr_positive[~np.eye(noise_corr_positive.shape[0], dtype=bool)])):.4f}")
    
    # Save results
    np.save(f"/root/capsule/scratch/long_time_scale_correlation/noise_corr_positive_{session_name}_{latent_var}_timewindow_{time_window0[0]}_{time_window0[1]}.npy", 
            noise_corr_positive)
    
except NameError as e:
    print(f"Variables not defined: {e}")
    print("Please ensure session_summary_df and row are defined")

In [ ]:
# Calculate noise correlation for negative units
print("Calculating noise correlation for negative units...")
try:
    # Filter for negative units
    negative_unit_index = row["negative_unit_index"].iloc[0]
    mask_negative = np.isin(psth_da.coords["unit_index"].values, negative_unit_index)
    psth_da_negative = psth_da.isel(unit=mask_negative)
    
    noise_corr_negative, residuals_negative, unit_residuals_negative = calculate_noise_correlation(
        psth_da=psth_da_negative,
        latent_var_values=df[latent_var][0],
        trial_numbers=trial_numbers,
        time_window=time_window0,
        align=align,
        include_trials=keep_ids
    )
    
    print(f"Noise correlation matrix shape (negative units): {noise_corr_negative.shape}")
    print(f"Mean absolute correlation (off-diagonal, negative): {np.mean(np.abs(noise_corr_negative[~np.eye(noise_corr_negative.shape[0], dtype=bool)])):.4f}")
    
    # Save results
    np.save(f"/root/capsule/scratch/long_time_scale_correlation/noise_corr_negative_{session_name}_{latent_var}_timewindow_{time_window0[0]}_{time_window0[1]}.npy", 
            noise_corr_negative)
    
except NameError as e:
    print(f"Variables not defined: {e}")
    print("Please ensure all required variables are loaded")

In [ ]:
# Visualization and analysis of noise correlations
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the plotting style
plt.style.use('default')
sns.set_palette("husl")

def plot_correlation_matrix(corr_matrix, title="Noise Correlation Matrix", vmin=-0.3, vmax=0.3):
    """Plot correlation matrix as a heatmap."""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Mask the diagonal
    mask = np.eye(corr_matrix.shape[0], dtype=bool)
    corr_masked = corr_matrix.copy()
    corr_masked[mask] = np.nan
    
    im = ax.imshow(corr_masked, cmap='RdBu_r', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('Unit Index')
    ax.set_ylabel('Unit Index')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Correlation Coefficient')
    
    return fig, ax

def analyze_correlation_distribution(corr_matrix, label=""):
    """Analyze the distribution of correlation coefficients."""
    # Extract off-diagonal elements
    mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
    off_diag_corr = corr_matrix[mask]
    
    # Remove NaN values
    off_diag_corr = off_diag_corr[~np.isnan(off_diag_corr)]
    
    print(f"\n{label} Correlation Statistics:")
    print(f"  Mean: {np.mean(off_diag_corr):.4f}")
    print(f"  Std:  {np.std(off_diag_corr):.4f}")
    print(f"  Median: {np.median(off_diag_corr):.4f}")
    print(f"  Min: {np.min(off_diag_corr):.4f}")
    print(f"  Max: {np.max(off_diag_corr):.4f}")
    print(f"  Mean absolute: {np.mean(np.abs(off_diag_corr)):.4f}")
    
    return off_diag_corr

# Try to plot and analyze results if they exist
try:
    if 'noise_corr_all' in locals():
        # Plot correlation matrices
        fig1, _ = plot_correlation_matrix(noise_corr_all, "All Units - Noise Correlation")
        plt.show()
        
        # Analyze distributions
        all_corr_dist = analyze_correlation_distribution(noise_corr_all, "All Units")
    
    if 'noise_corr_positive' in locals():
        fig2, _ = plot_correlation_matrix(noise_corr_positive, "Positive Units - Noise Correlation")
        plt.show()
        
        pos_corr_dist = analyze_correlation_distribution(noise_corr_positive, "Positive Units")
    
    if 'noise_corr_negative' in locals():
        fig3, _ = plot_correlation_matrix(noise_corr_negative, "Negative Units - Noise Correlation")
        plt.show()
        
        neg_corr_dist = analyze_correlation_distribution(noise_corr_negative, "Negative Units")
        
    # Compare distributions if multiple exist
    if all(['noise_corr_positive' in locals(), 'noise_corr_negative' in locals()]):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.hist(pos_corr_dist, bins=30, alpha=0.7, label='Positive Units', density=True)
        ax.hist(neg_corr_dist, bins=30, alpha=0.7, label='Negative Units', density=True)
        ax.set_xlabel('Correlation Coefficient')
        ax.set_ylabel('Density')
        ax.set_title('Distribution of Noise Correlations: Positive vs Negative Units')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()
        
except NameError:
    print("Correlation matrices not calculated yet. Please run the previous cells first.")

In [ ]:
# Complete analysis combining your TDR code with noise correlation
# This cell integrates your original code with the noise correlation analysis

# First, let's define the variables from your original code
try:
    # Variables from your original code (you can uncomment and modify as needed)
    # session_name = session_summary_df['session_name'][4]
    # row = session_summary_df.loc[session_summary_df["session_name"] == session_name]
    # positive_unit_index = row["positive_unit_index"].iloc[0]
    # negative_unit_index = row["negative_unit_index"].iloc[0]
    
    # binsize = '0.2'
    # align = "go_cue"
    # time_window0 = [-1, 0]
    # latent_var = 'ForagingCompareThreshold-value-1'
    
    # # Load data
    # psth_da = load_zarr(f"/root/capsule/scratch/psth_results/{session_name}_{binsize}s.zarr")
    # df = smart_read_csv(f"/root/capsule/scratch/behavior_summary/behavior_summary-{session_name}.csv")
    # keep_ids = np.asarray(df['response_trials'][0], dtype=int)
    
    # Now calculate noise correlations
    print("=== Noise Correlation Analysis ===")
    print("This analysis regresses out:")
    print("1. Correlation with latent variable (ForagingCompareThreshold-value-1)")
    print("2. Correlation with trial number")
    print("3. Subtracts the mean firing rate")
    print("4. Concatenates data from multiple trials within the time window")
    print()
    
    # The noise correlation function does the following:
    # 1. Selects the specified time window
    # 2. Filters for specified trials
    # 3. Concatenates firing rates across time points within trials
    # 4. Creates regressors for latent variable and trial number
    # 5. Performs linear regression to remove these confounds
    # 6. Calculates residuals (noise)
    # 7. Mean-centers the residuals
    # 8. Computes correlation matrix between units
    
    print("To run this analysis:")
    print("1. Define session_summary_df with your session data")
    print("2. Ensure your data paths are correct")
    print("3. Run the cells above in sequence")
    print("4. The results will be saved in /root/capsule/scratch/long_time_scale_correlation/")
    
except Exception as e:
    print(f"Note: This is a template cell. To run the actual analysis:")
    print("1. Uncomment and modify the variable definitions above")
    print("2. Ensure your session_summary_df is loaded")
    print("3. Update file paths as needed")
    print("4. Run all cells in sequence")

# Summary of outputs:
print("\n=== Expected Outputs ===")
print("1. noise_corr_all: Correlation matrix for all units")
print("2. noise_corr_positive: Correlation matrix for positive units")
print("3. noise_corr_negative: Correlation matrix for negative units")
print("4. Visualization plots showing correlation matrices and distributions")
print("5. Saved .npy files with correlation matrices")